# Día 7 — Pipelines, forecasting y comunicación

Cerrar el flujo reproducible y presentar una conclusión útil, honesta y verificable.

## Objetivos de aprendizaje

- Automatizar transformaciones y modelo en un Pipeline.
- Crear lags sin usar información futura.
- Separar Train/Validation/Test cronológicamente y comparar baselines.
- Comunicar evidencia, limitaciones y una solución reproducible.

> **Ruta presencial (3 horas):** origen, rezagos y disponibilidad 0–45 min; corte cronológico, baselines y Pipeline 45–90; evaluación 90–110; **práctica final de notebook y retroalimentación escrita** 110–180. Los diagnósticos extensos permanecen explicados para autoestudio. No se requiere exposición oral.

**Dataset:** `datos/macro_mensual_demo.csv`, serie sintética didáctica explícitamente no oficial. Se carga desde disco; no se consulta ningún servicio externo.

## Del notebook exploratorio al flujo reproducible

Un análisis manual puede transformar Train con una regla y Test con otra, o calcular una media usando todo el futuro. Un pipeline registra y aplica los mismos pasos en orden. En forecasting además importa **cuándo** existía cada dato: no basta con que una columna esté en el CSV si se publica después de la fecha que intentamos pronosticar.

Este ejemplo es mensual, de un paso adelante, didáctico y sintético. Aprenderemos el procedimiento, no la economía ecuatoriana.

### Vocabulario de series temporales

- **Frecuencia:** cada cuánto se observa la serie (mes, trimestre, año).
- **Horizonte:** cuántos periodos hacia adelante se pronostica; aquí un mes.
- **Lag:** valor rezagado, conocido del pasado.
- **Rolling window:** resumen de una ventana pasada.
- **Forecast origin:** fecha en que hacemos el pronóstico con información disponible hasta entonces.

Fijar estos conceptos determina qué filas se pueden entrenar y qué variables están disponibles.

> **Puente con vocabulario de econometría:** si tu formación es de estadística/econometría, es probable que ya conozcas estas ideas con otros nombres:
>
> | Aquí lo llamamos | En econometría/series de tiempo clásicas se suele llamar |
> |---|---|
> | Lag | Rezago (el mismo concepto que en un modelo AR) |
> | Rolling window | Ventana móvil / media móvil (suavizamiento, no el componente MA de un ARIMA) |
> | Forecast origin | Fecha de corte / origen del pronóstico |
> | Train / Validation / Test ordenados por fecha | Partición *in-sample* / *out-of-sample* (fuera de muestra) |
> | Baseline de persistencia (predecir el último valor conocido) | Caminata aleatoria (*random walk*) como modelo nulo |
>
> No usamos modelos ARIMA ni pruebas de estacionariedad en este curso — el enfoque aquí es predictivo, con regresión/árboles sobre variables de rezago — pero si esos términos te resultan más familiares, puedes traducir mentalmente con esta tabla.

## 1. Qué significa automatizar un análisis

Un Pipeline reúne transformaciones y estimador para que cada ajuste aprenda solo de Train. Aplicaremos ese principio a un flujo de pronóstico de un paso adelante: predecir el índice mensual de actividad con su historia reciente y variables macroeconómicas observadas. No mezclaremos pasado y futuro al validar.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def buscar_datos():
    candidatos = [Path("../datos"), Path("datos"), Path.cwd() / "datos", Path.cwd().parent / "datos"]
    for ruta in candidatos:
        if ruta.exists():
            return ruta.resolve()
    raise FileNotFoundError("No encuentro la carpeta datos/. Abre Jupyter desde la raíz o notebooks/ del repositorio.")

DATA_DIR = buscar_datos()
print("Datos locales:", DATA_DIR)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

serie = pd.read_csv(DATA_DIR / "macro_mensual_demo.csv", parse_dates=["fecha"])
serie = serie.sort_values("fecha").reset_index(drop=True)
serie.head()


### Conoce el archivo y la pregunta

Cada fila de `macro_mensual_demo.csv` representa un mes con índice de actividad, exportaciones, inflación y petróleo simulados. El target será el índice del siguiente mes bajo una formulación de fila t con predictores observados antes de t. Al explorar, confirma orden temporal, duplicados, huecos, unidades y nulos.

En una aplicación macroeconómica también registraríamos fecha de publicación y fecha de revisión: una estadística observada para enero puede publicarse en febrero y revisarse meses después.

### Serie temporal frente a tabla transversal

En una tabla transversal, normalmente se puede barajar filas si son independientes. En una serie, las filas vecinas comparten historia; barajar rompe el orden de predicción y puede colocar observaciones futuras en Train. Un score aleatorio respondería otra pregunta: reconstruir valores intercalados, no pronosticar un futuro aún no observado.

La visualización y el split deben preservar el calendario antes de decidir cualquier modelo.

### Procedencia y límites de estos datos

`macro_mensual_demo.csv` es una serie **sintética didáctica** creada para practicar flujo temporal; no son cifras oficiales, no describen la economía ecuatoriana real ni sirven para decisiones. La usamos porque 144 meses permiten demostrar lags y evaluación temporal en una clase breve. Un proyecto aplicado debe sustituirla por una serie pública local descargada, versionada y documentada, nunca conectarse a una API durante la clase.

In [ ]:
print("Dimensiones:", serie.shape)
print("Rango:", serie.fecha.min(), "a", serie.fecha.max())
print("Duplicados de fecha:", serie.fecha.duplicated().sum())
print("Nulos:")
print(serie.isna().sum())
plt.figure(figsize=(10,4))
plt.plot(serie.fecha, serie.indice_actividad)
plt.title("Índice mensual de actividad — serie sintética didáctica")
plt.ylabel("Índice (unidad didáctica)"); plt.xlabel("Fecha"); plt.grid(alpha=.2)


### ¿Qué muestra y qué no muestra la gráfica?

Una tendencia, repetición o salto puede sugerir comportamiento, pero 144 valores simulados no prueban estacionalidad real ni causalidad con petróleo. Busca orden, frecuencia constante y observaciones extremas. No suavices ni elimines un salto solo porque molesta: primero pregunta si es error de captura o un cambio posible del proceso.

### Revisar frecuencia antes de crear `shift`

`shift(1)` significa fila previa, no «un mes antes» por definición. Si hay dos filas para marzo o falta abril, los rezagos dejan de representar los meses esperados. Por eso ordenamos por fecha y comprobamos duplicados; en una serie pública revisaríamos también meses ausentes y si las magnitudes son flujos, stocks, índices o tasas.

Si hay huecos, decide cómo representarlos antes de generar lags: no interpoles automáticamente periodos sin datos.

## 2. Crear predictores conocidos en el origen y un target futuro

Cada fila se interpretará como un **origen de pronóstico** en el mes t: usamos información observada hasta t para pronosticar el índice del mes t+1. En consecuencia, `lag_1` será el índice observado en t; `lag_3`, el índice de t−2; y `lag_12`, el índice del mismo mes del año previo al target. La ventana móvil usa t, t−1 y t−2, todos conocidos al origen. El target futuro se crea por separado con `shift(-1)`.

## Lags: construir memoria sin copiar el futuro

La fila de marzo representa el **origen marzo** y su respuesta será abril. Al cierre de marzo ya conocemos `indice_actividad` de marzo, por lo que sirve como `lag_1`. Para este target de abril, el valor estacional comparable es abril del año previo, que se encuentra once filas antes del origen marzo. `lag_3` representa enero.

La media móvil puede incluir marzo, porque ese dato ya se observó en el origen. En cambio el target de abril se desplaza con `shift(-1)` y nunca se utiliza para calcular features de marzo. Así se evita fuga temporal.

> **Nota para quien viene de estadística:** `.shift()`, `.rolling()` y `.dt` son los tres métodos de Pandas para construir variables de rezago y ventanas móviles — se explican con un ejemplo mínimo en el notebook complementario [Día 0.1, Sección 9.5](00_1_Dia0_Puente_a_Scikit_Learn.ipynb). Aquí se usan varias veces seguidas para construir, a la vez, el objetivo a predecir (`shift(-1)`, un paso hacia adelante) y las variables de entrada (`shift(2)`, `shift(11)`, `rolling(3)`, todas hacia atrás, porque son las únicas que se conocerían en el momento real de predecir).

In [ ]:
s = serie.copy()
# La fila t es el origen; la respuesta a pronosticar es el índice de t+1.
s["target_siguiente_mes"] = s.indice_actividad.shift(-1)
s["fecha_objetivo"] = s.fecha.shift(-1)
# Features conocidas en t: dato actual, rezagos y media hasta el origen.
s["lag_1"] = s.indice_actividad
s["lag_3"] = s.indice_actividad.shift(2)
s["lag_12"] = s.indice_actividad.shift(11)
s["media_3_hasta_origen"] = s.indice_actividad.rolling(3).mean()
s["mes"] = s.fecha.dt.month
s["anio"] = s.fecha.dt.year
s = s.dropna().reset_index(drop=True)
features = ["precio_petroleo_usd", "inflacion_interanual_pct", "exportaciones_mill",
            "lag_1", "lag_3", "lag_12", "media_3_hasta_origen", "mes", "anio"]
target = "target_siguiente_mes"
s = s.dropna(subset=features + [target]).reset_index(drop=True)
print("Filas después de crear historia y target futuro:", len(s))
s[["fecha", "fecha_objetivo", "indice_actividad", target, "lag_1", "lag_12"]].head()


### Interpreta una fila con calendario

Supón que estamos al cierre de marzo y queremos pronosticar abril: `lag_1` es marzo, `lag_3` enero y `lag_12` abril del año pasado. Si falta un mes, los rezagos por fila dejarán de coincidir con los meses del calendario; por eso comprobamos frecuencia y huecos antes. La codificación de `mes` indica el mes del origen y puede ser cíclica en un modelo lineal.

Las primeras filas no tienen suficiente historia para `lag_12`, y la última no tiene respuesta futura; esas filas se descartan. En cada fila conservamos tanto `fecha` (origen) como `fecha_objetivo` (mes pronosticado).

In [ ]:
# Comprueba que las variables del origen corresponden a las fechas esperadas
fila = s.iloc[12]
print("Fecha origen:", fila.fecha.date())
print("Fecha objetivo:", fila.fecha_objetivo.date())
print("lag_1 guardado:", fila.lag_1)
print("índice observado en el origen:", fila.indice_actividad)
print("lag_3 guardado:", fila.lag_3)
print("lag_12 guardado:", fila.lag_12)
print("target futuro:", fila[target])
assert fila.lag_1 == fila.indice_actividad
assert fila.lag_3 == s.iloc[10].indice_actividad
assert fila.lag_12 == s.iloc[1].indice_actividad
assert fila[target] == s.iloc[13].indice_actividad


La comprobación `assert` convierte una idea de calendario en una verificación concreta. Si falla, detén el análisis y revisa orden, huecos y cómo se crearon los rezagos. Las ventanas se calculan en filas; no conocen fechas por sí solas.

### Verifica una fila a mano

La fecha de la fila es el origen; la fecha objetivo es un mes después. Verificamos que `lag_1` sea el valor observado en el origen, `lag_3` sea dos meses anterior al origen, `lag_12` corresponda al mismo mes del año previo a la respuesta y el target coincida con el siguiente mes. Las variables externas se asumen disponibles en el origen; en una aplicación real se auditarían fechas de publicación y revisiones.

### Covariables disponibles en el origen del pronóstico

El ejemplo incluye petróleo, inflación y exportaciones del mes de origen y asume que sus valores ya están disponibles al cierre de ese mes. Esa suposición puede ser falsa en tiempo real. Si una cifra se publica después, estaríamos usando información futura con respecto al origen, no haciendo un pronóstico operacional válido.

En un proyecto real se debe definir el origen de pronóstico, el rezago de publicación, revisiones/vintages y si una variable externa se conoce con antelación o también debe pronosticarse.

### Conjunto de información en cada fecha

En un origen t, un pronóstico válido usa solo la información disponible hasta t. Escribe una lista de variables y su fecha de disponibilidad: petróleo podría conocerse diariamente; exportaciones oficiales pueden publicarse semanas después; crecimiento trimestral puede revisarse. Un CSV retrospectivo suele mostrar el valor final revisado, no necesariamente la cifra disponible en el momento histórico.

Esta diferencia entre fecha de referencia y fecha de publicación es crítica para pronósticos macroeconómicos reales.

## 3. Separación temporal y baseline

Reservamos el 20 % final como Test, manteniendo el orden. Para seleccionar configuraciones, dividimos el periodo previo de forma cronológica en Train y Validation. Un baseline `lag_1` dice que el siguiente mes será igual al último observado; un baseline estacional usa `lag_12`. Son referencias obligatorias.

## Separación cronológica: futuro fuera de Train

Dividimos por fecha, no barajamos. El 20 % final se reserva como Test; dentro del 80 % inicial, el 20 % final funciona como Validation. En términos aproximados queda 64 % Train, 16 % Validation y 20 % Test del dataset transformado. Validation simula un periodo posterior a Train; Test, un periodo aún más futuro.

El orden reduce fuga temporal, pero una única fecha de corte puede coincidir con una fase económica particular. En series aplicadas se suelen comparar varios orígenes de pronóstico móviles (*rolling-origin evaluation*).

### Lectura temporal del split

Train va primero, Validation después y Test al final. Los límites de fecha impresos sirven para verificar el orden; las filas iniciales también se redujeron al crear lags. Validation no debe solaparse temporalmente con Train ni Test con ambos. Una separación temporal preserva el orden, pero no elimina autocorrelación ni cambios de régimen.

In [ ]:
corte = int(len(s) * .80)
dev, test = s.iloc[:corte].copy(), s.iloc[corte:].copy()
corte_val = int(len(dev) * .80)
train, val = dev.iloc[:corte_val].copy(), dev.iloc[corte_val:].copy()
print('Train | orígenes:', train.fecha.min().date(), '–', train.fecha.max().date(), '| target final:', train.fecha_objetivo.max().date(), '| filas:', len(train))
print('Validation | orígenes:', val.fecha.min().date(), '–', val.fecha.max().date(), '| target final:', val.fecha_objetivo.max().date(), '| filas:', len(val))
print('Test | orígenes:', test.fecha.min().date(), '–', test.fecha.max().date(), '| targets:', test.fecha_objetivo.min().date(), '–', test.fecha_objetivo.max().date(), '| filas:', len(test))
assert train.fecha_objetivo.max() <= val.fecha.min()
assert val.fecha_objetivo.max() <= test.fecha.min()


## Baselines: reglas que un modelo debe superar

Persistencia `lag_1` pronostica que el próximo mes será igual al índice observado en el origen. El baseline estacional `lag_12` usa el valor del mes objetivo un año antes (para origen marzo → objetivo abril, toma abril del año anterior). Son reglas simples, transparentes y difíciles de superar en algunas series.

Calculamos MAE en Validation para seleccionar; no en Test. MAE es promedio de $|y-\hat{y}|$ en unidades del índice sintético. Un baseline puede ser la decisión final adecuada si Random Forest añade complejidad sin bajar el error de forma convincente.

In [ ]:
def mae(y, p): return mean_absolute_error(y, p)
for nombre, col in [("Persistencia lag 1", "lag_1"), ("Estacional lag 12", "lag_12")]:
    print(nombre, "MAE Validation:", round(mae(val[target], val[col]), 3))


### Comparar baselines con el patrón de la serie

Si el índice cambia lentamente, persistencia puede funcionar bien. Si el comportamiento se repite cada doce meses, lag estacional puede ser más competitivo. El menor MAE observado no prueba cuál regla será mejor en cualquier periodo; describe Validation.

Antes de entrenar el Pipeline, anota cuál baseline gana y cuánto margen de error tiene. Después compara Random Forest con ese resultado, no con cero.

### Un baseline puede contener mucho conocimiento

La persistencia expresa una hipótesis concreta: no hay cambio de un periodo al siguiente. La estacionalidad supone que se repite un patrón anual. Estas reglas capturan inercia o calendario sin aprender muchas divisiones; por eso son competidores serios, no modelos «tontos» que se incluyen por requisito.

Si el baseline gana, pregunta si las covariables aportan nueva información disponible en el origen o si solo repiten historia que los lags ya representan.

### Interpreta el baseline temporal

Compara los dos MAE y considera el patrón mensual. Menor MAE es mejor en la unidad sintética del índice. El baseline estacional requiere doce meses de historia; el de persistencia puede responder mejor ante series suaves. Ninguno usa covariables externas: un modelo más complejo debe superar una referencia razonable fuera de muestra.

## 4. Pipeline y selección pequeña de modelos

Las covariables aquí son numéricas. El Pipeline incluye imputación mediana, escalado y un regresor; Random Forest no necesita escalado, pero lo mantenemos para ilustrar el flujo automatizado reutilizable. Ajustaremos pocas configuraciones dentro del desarrollo y elegiremos con Validation, no con Test.

## Dentro de un Pipeline, qué aprende cada paso

`ColumnTransformer` enruta columnas hacia su transformador. `SimpleImputer(strategy='median')` calcula medianas en Train; `StandardScaler` calcula medias y escalas en Train; el estimador Random Forest aprende reglas de división. `fit()` aprende esos valores y parámetros. `predict()` aplica las medianas/escalas ya aprendidas, no las recalcula con Validation.

Aunque los árboles no necesitan escala, la dejamos para hacer visible el flujo modular que luego se puede reutilizar con KNN o SVM. Si cambiamos el estimador, mantenemos intacta la definición de preparación.

### Entrenar, transformar y predecir

Para un dato nuevo, el Pipeline aplica en orden imputación y escalado usando valores aprendidos durante `.fit()`, después entrega el vector transformado al estimador. Guardar modelo y transformadores juntos evita olvidar una mediana o estándar al reproducir una predicción. El Pipeline facilita auditoría, pero todavía se debe versionar el dataset y el horizonte.

> **Antes de correr esta celda — desarmando el Pipeline anidado:** léelo de adentro hacia afuera. `Pipeline([("imputer", ...), ("scale", ...)])` arma un mini-pipeline que primero imputa faltantes y luego escala — ese mini-pipeline se usa como transformador de columnas numéricas dentro de `ColumnTransformer([("num", mini_pipeline, features)])` — y ese `ColumnTransformer` se usa, a su vez, como el primer paso (`"prep"`) de un `Pipeline` más grande, cuyo segundo paso (`"model"`) es el `RandomForestRegressor`. Tres niveles de anidamiento, pero cada uno es, individualmente, la misma receta `[(nombre, paso), ...]` del notebook complementario [Día 0.1, Sección 4](00_1_Dia0_Puente_a_Scikit_Learn.ipynb).

In [ ]:
pipe = Pipeline([
    ("prep", ColumnTransformer([("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                                                     ("scale", StandardScaler())]), features)])),
    ("model", RandomForestRegressor(random_state=42, n_estimators=200, min_samples_leaf=3))
])
pipe.fit(train[features], train[target])
pred_val = pipe.predict(val[features])
print("MAE RF Validation:", round(mae(val[target], pred_val), 3))
print("MAE baseline lag1:", round(mae(val[target], val.lag_1), 3))
print("MAE baseline lag12:", round(mae(val[target], val.lag_12), 3))


### Interpreta la validación antes de seleccionar

Compara MAE de Random Forest con persistencia y baseline estacional. Si pierde ante ambas reglas, selecciona la regla más simple. Si gana por una diferencia mínima, considera error de estimación, tamaño de la ventana y estabilidad antes de justificar su complejidad.

Un Pipeline garantiza reproducibilidad de transformaciones, pero no hace automáticamente correcto el diseño temporal ni soluciona variables publicadas tarde.

### Diagnóstico de Validation

¿El Pipeline supera ambos baselines? Un resultado peor no es un fracaso: evita justificar complejidad innecesaria. Inspecciona errores por fecha y revisa si hay cambios de régimen. Con 144 observaciones y una serie sintética, los errores y la selección son ilustrativos; no son una evaluación económica formal.

## Regla de decisión, fijada antes de ver Test

La celda elige Random Forest solo si su MAE de Validation es menor que el de ambos baselines. En caso contrario elige el baseline con menor MAE. Es una regla sencilla y auditable; otro uso podría penalizar más errores extremos, usar RMSE o definir costos asimétricos, pero esa regla también debe quedar escrita antes de medir Test.

Test no elige hiperparámetros ni horizonte. Evalúa una decisión que ya cerramos sobre los datos de desarrollo.

## 5. Decisión final y Test

Antes de ejecutar, declara una regla simple: elegir Random Forest solo si mejora el menor MAE de los baselines en Validation; de otro modo elegir el baseline ganador. Fijamos la regla y evaluamos una sola vez en el tramo final.

> **Nota para quien viene de estadística:** la línea de `elegido = "rf" if ... else (... if ... else ...)` es una expresión ternaria (Día 0.1, Sección 9.1) **anidada dentro de otra**: primero decide si el Random Forest ganó; si no, decide entre las dos líneas base. Para leerla, resuelve primero la condición más externa (`mae_rf_val < min(...)`): si es verdadera, `elegido = "rf"` y ya terminaste; si es falsa, recién ahí evalúas el `if`/`else` que está entre paréntesis.

In [ ]:
mae_rf_val = mae(val[target], pred_val)
mae_lag1_val = mae(val[target], val.lag_1)
mae_lag12_val = mae(val[target], val.lag_12)
elegido = "rf" if mae_rf_val < min(mae_lag1_val, mae_lag12_val) else ("lag_1" if mae_lag1_val <= mae_lag12_val else "lag_12")
if elegido == "rf":
    pred_test = pipe.predict(test[features])
else:
    pred_test = test[elegido].to_numpy()
pd.DataFrame([{"modelo":elegido, "MAE_validation":min(mae_rf_val,mae_lag1_val,mae_lag12_val),
               "MAE_test":mae(test[target],pred_test),
               "RMSE_test":mean_squared_error(test[target],pred_test)**.5}]).set_index("modelo").round(3)


### Por qué este Test es de un paso adelante

En cada fila de Test, `fecha` es el origen y `fecha_objetivo` el mes siguiente. `lag_1` es el índice ya observado en el origen; al avanzar un mes se incorpora el valor real recién observado para el próximo pronóstico. Esto evalúa pronósticos **rolling de un paso** con actualización mensual.

Para pronosticar doce meses de una vez no conoceríamos los targets intermedios; habría que alimentar predicciones previas al siguiente paso (pronóstico recursivo) o construir modelos directos por horizonte. No confundas el buen resultado de un paso con la precisión a doce meses.

### Errores sistemáticos frente a errores aleatorios

Un error cercano a cero en promedio puede esconder subpronósticos en unos periodos y sobrepronósticos en otros. Mira el gráfico de error por fecha: rachas del mismo signo sugieren sesgo temporal; picos aislados sugieren episodios difíciles; cambios de dispersión pueden indicar un nuevo régimen. Resume un hallazgo con fecha y magnitud, no solo con «el modelo se equivocó».

### MAE y RMSE cuentan historias diferentes

MAE expresa el tamaño típico del error absoluto y es fácil de leer en la unidad del índice. RMSE eleva errores al cuadrado antes de promediar, así que un periodo con error extremo pesa más. Si ambos son cercanos, no hay errores enormes dominando; si RMSE es mucho mayor, inspecciona fechas de error elevado y cambios de nivel.

In [ ]:
# Diagnóstico temporal de errores en el Test final
errores_test = pd.DataFrame({
    "fecha_objetivo": test.fecha_objetivo.to_numpy(),
    "observado": test[target].to_numpy(),
    "pronostico": pred_test
})
errores_test["error"] = errores_test.observado - errores_test.pronostico
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(errores_test.fecha_objetivo, errores_test.error, marker="o")
ax.axhline(0, color="black", linewidth=1)
ax.set(title="Error observado − pronóstico en Test", xlabel="Fecha", ylabel="Error (unidad sintética)")
ax.grid(alpha=.2)
plt.tight_layout()


### Qué buscar en los residuos

Una secuencia de errores mayormente positiva sugiere subpronóstico; negativa, sobrepronóstico. Grupos de errores grandes o patrones por fecha pueden indicar cambios de régimen, estacionalidad no capturada o covariables no disponibles. No uses este gráfico para ajustar repetidamente el modelo y luego afirmar que Test siguió siendo independiente.

Como la serie es sintética, la inspección es una práctica de diagnóstico; no presenta una anomalía económica observada.

### Comunica el pronóstico

MAE es el desvío absoluto promedio en unidades del índice; RMSE penaliza más los errores extremos. Compara ambos con Validation, mira fechas donde el modelo falla y explica si el método superó baselines. Si el test final es peor, puede indicar variación temporal; no reajustes mirando repetidamente ese periodo. La decisión automática del ejemplo hace explícita la regla de selección, no una garantía de óptimo.

### De métrica a mensaje ejecutivo

Una **síntesis escrita para analistas** debe responder en lenguaje llano: qué pregunta se abordó, cuál fue el baseline, cuánto error hubo, qué periodos fueron difíciles, qué información estaba disponible en cada fecha y qué datos faltan para confiar en el resultado. El mejor algoritmo no reemplaza el juicio sobre la revisión de cifras, frecuencia ni propósito.

La serie usada aquí es simulada. En un proyecto del BCE no la citarías como evidencia ni desplegarías un modelo sin aprobación y validación institucional.

## 6. Comunicación escrita de resultados y proyecto integrador

El bloque final de 70 minutos se dedica a **completar el notebook y recibir retroalimentación**, sin turnos de exposición. **Práctica final del Proyecto Integrador (40 puntos):** reúne en el mismo notebook las tres tareas guiadas de 20 puntos de los días 4, 5 y 6, sin cobrarles una nota nueva. En una **síntesis escrita con una figura o tabla**, explica por qué elegiste regresión, clasificación o clustering; qué datos y transformaciones utilizaste; qué alternativas comparaste en Validation; qué decisión fijaste antes de Test; qué ocurrió en la comprobación final; y qué límites impiden prometer resultados futuros. La división cronológica es necesaria para forecasting, pero no se impone a todos los problemas. En clustering no hay una clase real con la que calcular Accuracy: se comprueban perfiles/estabilidad en las filas retenidas.

### Rúbrica de la práctica final del Proyecto Integrador (40 puntos)

- **Comprobación final reservada (12):** método/regla elegidos antes de ver Test; evaluación de Test una vez con MAE/RMSE/R² o matriz/Recall/Precision/F1 según el problema. En clustering, perfil y estabilidad de las filas retenidas sin inventar `y` ni Accuracy.
- **Interpretación frente a la referencia (10):** comparar resultado final con baseline/alternativas ya vistos en Validation; explicar conteos de errores, unidades y/o tamaños/centroides y si la diferencia importa para la pregunta.
- **Notebook integrado reproducible (8):** Tareas 1–3 incorporadas en orden, lectura del CSV local, transformación aprendida solo con Train, código ejecutable de principio a fin y salidas coherentes con el protocolo. Aquí se evalúa la **integración final**, no se vuelve a puntuar por separado cada gráfico o limpieza ya calificados.
- **Comunicación ejecutiva escrita (6):** redactar dentro del notebook una síntesis breve de pregunta, tipo, fuente, método, evidencia y decisión con una figura/tabla legible; explicar qué significa el resultado para un público no técnico. Sin exposición oral.
- **Límites y siguiente paso (4):** identificar al menos dos límites concretos (muestra, dependencia temporal/por país, desbalance, fuente sintética, disponibilidad de variables) y una comprobación futura sensata sin afirmaciones causales indebidas.

**Total de esta práctica: 40 puntos.** Las tres tareas guiadas anteriores valen **20 cada una** (60 formativos) y la diagnóstica del Día 1 vale 0. El instrumento detallado y la escala de calificación están en [`INSTRUMENTOS_EVALUACION_MODULO3.md`](../INSTRUMENTOS_EVALUACION_MODULO3.md).

### Cómo defender la entrega final

La práctica final suma 40 puntos. Entrega **el mismo notebook** que comenzó con la Tarea 1, con los tres avances formativos integrados y una **conclusión escrita nueva** basada en Test o en estabilidad de perfiles retenidos. La rúbrica final mide esa comprobación, su interpretación, reproducibilidad y comunicación escrita; no vuelve a asignar los 20 puntos ya otorgados por cada avance. **No se requiere exposición oral ni diapositivas.** En regresión/clasificación distingue Train, Validation y Test; en clustering documenta el protocolo de tres partes sin exigir `y` ni Accuracy. Las categorías pertinentes se identifican desde la exploración, pero su codificación se aprende solo con Train.

Una puntuación alta no significa que el pronóstico sea implementable automáticamente: autorización, seguimiento, actualizaciones y costos operativos exceden la métrica del ejercicio.

## Actividad autónoma

Cambia la regla de selección para optimizar una métrica distinta o define un horizonte de dos meses, creando los targets sin que los lags incluyan futuro. Explica cómo cambian los baselines, el tamaño de la muestra y la interpretación. No presentes datos sintéticos como datos del BCE.